# NISAR GUNW to pairwise dSWE

The default path is a software and adapter demonstration using a small bundled ASF prelaunch GUNW crop. It reads the product and uses native ellipsoid incidence so the notebook runs without credentials or external files. Its calculated dSWE is not a snow-science result.

For actual NISAR snow analysis, place one production GUNW in `data/external/nisar/gunw/` and its matching NISAR-modified Copernicus DEM in `data/external/nisar/dem/`. The notebook finds one file in each folder automatically. When a folder has multiple files, set `SNOWIN_GUNW` or `SNOWIN_NISAR_DEM` to the exact path. The production path computes terrain-local incidence; the bundled sample's native ellipsoid incidence is a distinct geometry calculation. See the [NISAR DEM guidance](../docs/vertical_datums.md).

Install the optional integrations and notebook tools with `python -m pip install -e ".[nisar,geometry,dask]" --group notebooks`.

To acquire a production GUNW, use the guarded [search and download notebook](03_nisar_pytools_search_and_snowin.ipynb) or `scripts/download_nisar_gunw.py`.

In [ ]:
import os
from pathlib import Path

from IPython.display import display

from snowin import compute_leinss_dswe
from snowin.io import add_gunw_incidence, open_gunw

working_path = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in (working_path, *working_path.parents)
        if (candidate / "src" / "snowin").is_dir()
    ),
    working_path,
)
example_gunw = repo_root / "notebooks" / "data" / "nisar_sample_gunw_crop.h5"
gunw_dir = repo_root / "data" / "external" / "nisar" / "gunw"
dem_dir = repo_root / "data" / "external" / "nisar" / "dem"

configured_gunw = os.environ.get("SNOWIN_GUNW")
downloaded_gunws = sorted(gunw_dir.glob("*.h5")) if gunw_dir.is_dir() else []
if configured_gunw:
    gunw_path = Path(configured_gunw).expanduser().resolve()
elif len(downloaded_gunws) == 1:
    gunw_path = downloaded_gunws[0].resolve()
elif len(downloaded_gunws) > 1:
    raise ValueError("Set SNOWIN_GUNW to select one of the downloaded GUNW products.")
else:
    gunw_path = example_gunw.resolve()

configured_dem = os.environ.get("SNOWIN_NISAR_DEM")
dem_candidates = (
    sorted([*dem_dir.glob("*.tif"), *dem_dir.glob("*.tiff")])
    if dem_dir.is_dir()
    else []
)
if configured_dem:
    dem_path = Path(configured_dem).expanduser().resolve()
elif len(dem_candidates) == 1:
    dem_path = dem_candidates[0].resolve()
elif len(dem_candidates) > 1:
    raise ValueError(
        "Set SNOWIN_NISAR_DEM to select the DEM matching this GUNW product."
    )
else:
    dem_path = None

if dem_path is not None and not configured_gunw and gunw_path == example_gunw.resolve():
    raise ValueError(
        "Set SNOWIN_GUNW as well so the prepared DEM is paired with its GUNW."
    )
if not gunw_path.is_file():
    raise FileNotFoundError(gunw_path)
if dem_path is not None and not dem_path.is_file():
    raise FileNotFoundError(dem_path)

print("GUNW:", gunw_path)
print("prepared DEM:", dem_path if dem_path is not None else "not configured")
print(
    "incidence mode:",
    "local terrain incidence from the prepared DEM"
    if dem_path is not None
    else "native GUNW ellipsoid incidence",
)

## 1. Read phase and source metadata

open_gunw() converts the declared source phase direction to the SnowIn convention and reads wavelength metadata. It does **not** calculate incidence. The returned xarray Dataset records the state value phase_normalized_product. Correction fields, including native radar-grid screens, are available with source metadata and are not applied automatically.

In [ ]:
pair = open_gunw(gunw_path, chunks="auto")
assert pair.attrs["snowin_data_state"] == "phase_normalized_product"
print(
    {
        "data_state": pair.attrs["snowin_data_state"],
        "phase_definition": pair.phase.attrs["phase_difference_definition"],
        "source_phase_definition": pair.attrs["source_phase_difference_definition"],
        "reference_time": pair.attrs["reference_time"],
        "secondary_time": pair.attrs["secondary_time"],
        "temporal_edge": pair.attrs["temporal_edge"],
        "wavelength_m": pair.attrs["wavelength_m"],
        "wavelength_source": pair.attrs["wavelength_source"],
    }
)
display(pair)

## 2. Add incidence and retrieve pairwise dSWE

With the bundled sample, this cell reads the GUNW's native ellipsoid-referenced incidence field. If `SNOWIN_GUNW` and its matching `SNOWIN_NISAR_DEM` are set, it instead computes terrain-local incidence from the prepared DEM and radar-grid look vectors. The selected incidence reference is printed so the two paths remain explicit.


In [ ]:
try:
    if dem_path is None:
        pair = add_gunw_incidence(
            pair,
            gunw_path,
            incidence_source="product_ellipsoid",
        )
    else:
        pair = add_gunw_incidence(
            pair,
            gunw_path,
            dem=dem_path,
            dem_source="nisar_cop30",
            require_vertical_datum_match=True,
        )

    assert pair.attrs["snowin_data_state"] == "retrieval_ready_pair"
    assert (
        pair.phase.attrs["phase_difference_definition"]
        == "reference_minus_secondary"
    )
    assert pair.incidence_angle.attrs["incidence_angle_reference"] in {
        "ellipsoid",
        "local",
    }

    print(
        {
            "data_state": pair.attrs["snowin_data_state"],
            "incidence_reference": pair.incidence_angle.attrs[
                "incidence_angle_reference"
            ],
            "incidence_source": pair.attrs["incidence_angle_source"],
            "geometry_supported_fraction": float(
                pair.geometry_valid.mean().compute()
            ),
            "vertical_datum_status": pair.attrs.get(
                "vertical_datum_status", "not applicable to ellipsoid incidence"
            ),
        }
    )
    display(pair[["phase", "incidence_angle", "geometry_valid"]])

    dswe = compute_leinss_dswe(
        pair.phase.where(pair.geometry_valid),
        pair.incidence_angle.where(pair.geometry_valid),
        wavelength_m=float(pair.attrs["wavelength_m"]),
    )
    display(dswe)
    print("pairwise dSWE units:", dswe.attrs["units"])
finally:
    pair.close()
